Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 07 — dostosowanie modelu pretrained (EuroSAT)

Materiał towarzyszący [`README.pl.md`](README.pl.md). Inny zbiór danych niż wykład (EuroSAT,
zdjęcia satelitarne, nie Caltech101) i inne pytanie: czy transfer learning z ImageNet
pomaga, gdy domena jest wizualnie odległa, i czy częściowy fine-tuning daje tu coś
więcej niż sam zamrożony backbone.


In [ ]:
import contextlib
import io
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.datasets as tvd
from torchvision.models import ResNet18_Weights, resnet18  # noqa: F401 - uzywane w TODO ponizej

torch.manual_seed(0)
torch.set_num_threads(4)  # stabilnosc watkow BLAS pod jupyter/ipykernel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Uzywane urzadzenie: {device}")

## 1. Dane: EuroSAT — struktura i eksploracja


In [ ]:
_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = (
    Path(os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache"))
    .expanduser()
    .resolve()
)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    full_ds = tvd.EuroSAT(root=DATA_CACHE_DIR, download=True)

CLASSES = full_ds.classes
print(f"liczba obrazow: {len(full_ds)}, liczba klas: {len(CLASSES)}")
print("klasy:", CLASSES)

fig, axes = plt.subplots(1, 10, figsize=(16, 2))
shown = set()
for img, label in full_ds:
    if label not in shown:
        axes[label].imshow(img)
        axes[label].set_title(CLASSES[label], fontsize=8)
        axes[label].axis("off")
        shown.add(label)
    if len(shown) == 10:
        break
plt.tight_layout()
plt.show()

**Zadanie:** czy te zdjęcia przypominają cokolwiek z ImageNet (zwierzęta, przedmioty,
sceny "z poziomu ziemi")? Zapisz zdaniem swoją wstępną intuicję, czy transfer learning
z ImageNet powinien tu pomóc.


_(Twoja odpowiedź tutaj.)_

## 2. Preprocessing zgodny z wagami pretrained

ResNet18 wymaga **dokładnie** takiego preprocessingu, jakiego użyto podczas treningu
na ImageNet — inaczej cechy wyuczone przez backbone nie będą miały sensu dla nowych
danych.


In [ ]:
def build_preprocess():
    # TODO: zwroc transformacje ZGODNA z wagami ResNet18_Weights.IMAGENET1K_V1
    # (podpowiedz: obiekt wag ma gotowa metode .transforms() - uzyj jej, nie
    # przepisuj recznie rozmiaru/normalizacji na piechote)
    raise NotImplementedError("TODO: build_preprocess")


preprocess = build_preprocess()
print(preprocess)

In [ ]:
sample_img, _ = full_ds[0]
sample_tensor = preprocess(sample_img)
print(f"ksztalt po preprocessingu: {tuple(sample_tensor.shape)}")
assert sample_tensor.shape == (3, 224, 224), "oczekiwano (3, 224, 224)"
print("OK: preprocessing zgodny z wejsciem ResNet18.")

## 3. Podział na train/val/test (gotowe)

50 obrazów/klasę (500 łącznie), podział 30/10/10 na train/val/test.


In [ ]:
PER_CLASS = 50
class_to_indices = {c: [] for c in range(10)}
for idx, (_, label) in enumerate(full_ds):
    if len(class_to_indices[label]) < PER_CLASS:
        class_to_indices[label].append(idx)
    if all(len(v) >= PER_CLASS for v in class_to_indices.values()):
        break

selected_indices = [idx for c in range(10) for idx in class_to_indices[c]]
selected_labels = [c for c in range(10) for _ in class_to_indices[c]]

imgs = [preprocess(full_ds[idx][0]) for idx in selected_indices]
X = torch.stack(imgs)
y = torch.tensor(selected_labels)
print(f"X={tuple(X.shape)}  y={tuple(y.shape)}")

g = torch.Generator().manual_seed(0)
train_idx, val_idx, test_idx = [], [], []
for c in range(10):
    positions = (y == c).nonzero().flatten()
    perm = positions[torch.randperm(len(positions), generator=g)]
    train_idx.append(perm[:30])
    val_idx.append(perm[30:40])
    test_idx.append(perm[40:50])
train_idx, val_idx, test_idx = torch.cat(train_idx), torch.cat(val_idx), torch.cat(test_idx)
assert len(set(train_idx.tolist()) & set(val_idx.tolist()) & set(test_idx.tolist())) == 0

X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
X_val, y_val = X[val_idx].to(device), y[val_idx].to(device)
X_test, y_test = X[test_idx].to(device), y[test_idx].to(device)
print(f"train={tuple(X_train.shape)}  val={tuple(X_val.shape)}  test={tuple(X_test.shape)}")

loss_fn = nn.CrossEntropyLoss()


def train_eval(model, param_groups, epochs, batch_size=16, seed=0):
    """Gotowa petla treningowa - Twoja decyzja to CO trafia do param_groups
    i w jakim trybie (train/eval) jest model PRZED wywolaniem tej funkcji."""
    import time

    torch.manual_seed(seed)
    opt = torch.optim.Adam(param_groups)
    n = X_train.shape[0]
    start = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(X_train[idx]), y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - start

    was_training = model.training
    model.eval()
    with torch.no_grad():
        val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        test_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
    if was_training:
        model.train()
    return {"val_acc": val_acc, "test_acc": test_acc, "train_time_s": train_time}

## 4. Wczytanie modelu pretrained i wymiana głowicy

Wczytaj `resnet18` z wagami `ResNet18_Weights.IMAGENET1K_V1` (przypięte explicite) i
zastąp `model.fc` nową warstwą `nn.Linear` o 10 wyjściach (klasy EuroSAT).


In [ ]:
def build_model(n_classes=10):
    # TODO:
    # 1. model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1) - wagi PRZYPIETE
    #    explicite, nie "DEFAULT"
    # 2. model.fc = nn.Linear(?, n_classes) - podmien glowice (in_features =
    #    oryginalny model.fc.in_features, ZANIM go nadpiszesz)
    # 3. return model.to(device)
    raise NotImplementedError("TODO: build_model")


m_check = build_model()
print("nowa glowica:", m_check.fc)
# TODO: zweryfikuj programowo (nie tylko czytajac kod):
# 1. m_check.fc.out_features == 10
# 2. ze model.fc po przypisaniu faktycznie wskazuje na nowa warstwe (sprawdz
#    np. in_features/out_features zamiast domyslac sie z samego kodu)
raise NotImplementedError("TODO: weryfikacja glowicy")

## 5. Zamrożenie backbone'u i trening samej głowicy

Zamroź wszystkie parametry backbone'u, zostaw trenowalną tylko głowicę. Pamiętaj o
kolejności: najpierw wymiana głowicy (sekcja 4), dopiero teraz zamrażanie. Ustaw
model w trybie, który **nie pozwoli dryfować statystykom BatchNorm** backbone'u.


In [ ]:
m_frozen = build_model()
# TODO:
# 1. zamroz WSZYSTKIE parametry backbone'u (requires_grad = False)
# 2. odmroz tylko parametry glowicy (fc)
# 3. ustaw model w trybie, ktory nie pozwoli BatchNorm backbone'u dryfowac
#    (zobacz wyklad, sekcja o BatchNorm - jeden konkretny tryb to zatrzymuje)
raise NotImplementedError("TODO: zamrozenie backbone'u")

before_backbone = {
    n: p.detach().clone() for n, p in m_frozen.named_parameters() if not n.startswith("fc")
}

# TODO: wywolaj train_eval z poprawna lista trenowalnych parametrow (tylko fc),
# epochs=15, zapisz wynik jako res_frozen
raise NotImplementedError("TODO: trening glowicy")

# TODO: zweryfikuj, ze KAZDY parametr backbone'u (wszystko poza "fc") jest
# identyczny jak w before_backbone (torch.equal) - jesli nie, cos jest zle
# zamrozone
raise NotImplementedError("TODO: weryfikacja zamrozenia")

## 6. Częściowy fine-tuning

Odmroź `layer4` razem z głowicą. Zbuduj optymalizator z **dwiema grupami parametrów**
o różnych learning rate.


**Zadanie:** jakie wartości LR wybrałaś/eś dla obu grup i dlaczego taki stosunek?

_(Twoja odpowiedź tutaj.)_

In [ ]:
m_partial = build_model()
# TODO:
# 1. zamroz wszystko, potem odmroz layer4 i fc
# 2. model.train() (tym razem CHCEMY, zeby BatchNorm w layer4 sie dostosowywal)
# 3. zbuduj param_groups: lista dwoch dict-ow {"params": ..., "lr": ...} -
#    osobny (mniejszy) LR dla layer4, osobny (wiekszy) dla fc
raise NotImplementedError("TODO: czesciowy fine-tuning - setup")

before_frozen_part = {
    n: p.detach().clone()
    for n, p in m_partial.named_parameters()
    if not n.startswith("layer4") and not n.startswith("fc")
}

# TODO: train_eval(m_partial, param_groups, epochs=8) -> res_partial
raise NotImplementedError("TODO: trening czesciowego fine-tuningu")

# TODO: zweryfikuj, ze WSZYSTKO poza layer4/fc jest niezmienione
raise NotImplementedError("TODO: weryfikacja czesciowego zamrozenia")

## 7. Zadanie diagnostyczne

Poniższy kod **nie działa poprawnie** — strata w ogóle nie maleje, accuracy zostaje na
poziomie przypadkowego zgadywania. Nie rzuca wyjątku, wygląda na poprawny.


In [ ]:
torch.manual_seed(0)
m_bug = build_model()
for p in m_bug.parameters():
    p.requires_grad = False
m_bug.fc.requires_grad_(True)
# optymalizator zbudowany TUTAJ, odwolujac sie do obecnej glowicy...
optimizer_bug = torch.optim.Adam(m_bug.fc.parameters(), lr=0.01)
# ...ale glowica jest podmieniana PO zbudowaniu optymalizatora:
m_bug.fc = nn.Linear(m_bug.fc.in_features, 10)
m_bug.eval()

losses = []
for _ in range(10):
    optimizer_bug.zero_grad()
    loss = loss_fn(m_bug(X_train[:16]), y_train[:16])
    loss.backward()
    optimizer_bug.step()
    losses.append(round(loss.item(), 4))
print("strata w kolejnych krokach:", losses)
print("model.fc.weight.requires_grad:", m_bug.fc.weight.requires_grad)

**Zadanie:** znajdź przyczynę, odwołując się do konkretnej linii kodu i do tego, co
faktycznie obserwujesz (strata, `requires_grad`, zawartość
`optimizer_bug.param_groups`). Nie wystarczy "coś jest nie tak z optymalizatorem".


In [ ]:
# TODO: zweryfikuj programowo swoja hipoteze - np. sprawdz, czy AKTUALNY
# m_bug.fc jest w ktorejs z grup optimizer_bug.param_groups (porownaj obiekty
# przez "is", nie przez wartosci)
raise NotImplementedError("TODO: weryfikacja diagnozy")

_(Twoja odpowiedź tutaj.)_

## 8. Porównanie strategii


In [ ]:
# TODO: wypisz tabele porownawcza res_frozen vs res_partial (val_acc, test_acc, czas)
raise NotImplementedError("TODO: podsumowanie porownania")

**Rozszerzenie opcjonalne:** zbadaj wpływ JEDNEGO ustawienia na wynik
częściowego fine-tuningu (inny stosunek LR albo inny zakres odmrażania).
Ten dodatkowy przebieg nie jest częścią rdzenia obowiązkowego.


In [ ]:
# TODO (rozszerzenie opcjonalne): zbuduj WARIANT czesciowego fine-tuningu
# rozny od sekcji 6 (np. inny
# stosunek LR, albo odmrozenie layer3+layer4 zamiast samego layer4), wytrenuj
# (train_eval), porownaj wynik z sekcja 6
raise NotImplementedError("TODO (rozszerzenie): wariant hiperparametrow")

_(Opcjonalna odpowiedź tutaj.)_

## 9. Confusion matrix i analiza błędów

Dla lepszej (według walidacji) strategii z sekcji 5/6.


In [ ]:
# TODO:
# 1. wybierz lepszy model wedlug walidacji (res_frozen vs res_partial)
# 2. policz predykcje na X_test, zbuduj confusion matrix 10x10
# 3. narysuj confusion matrix z etykietami klas
# 4. znajdz i wyswietl 6-8 zle sklasyfikowanych przykladow (obraz + prawdziwa/
#    przewidziana etykieta) - pamietaj o odwroceniu normalizacji do wyswietlenia
raise NotImplementedError("TODO: confusion matrix i analiza bledow")

_(Twoja odpowiedź tutaj.)_

## 10. Wnioski

Odpowiedz, uzasadniając rzeczywistymi liczbami z sekcji 5-9.


**1. Czy transfer learning z ImageNet rzeczywiście pomógł na danych tak odległych wizualnie? Jak duża jest różnica względem Twojej intuicji z sekcji 1?**

_(Twoja odpowiedź tutaj.)_

**2. Czy częściowy fine-tuning dał mierzalną korzyść względem zamrożonego backbone'u, czy tylko zwiększył koszt? Porównaj z wynikiem z wykładu (Caltech101).**

_(Twoja odpowiedź tutaj.)_

**3. Jakie jest co najmniej jedno ograniczenie tego eksperymentu, które nie pozwala uogólnić wniosków na "strategia X jest zawsze lepsza dla danych satelitarnych"?**

_(Twoja odpowiedź tutaj.)_